In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
from config import RAW, STATIC
from src.analysis.pipeline import build_prepared_dataset
from src.analysis.timewindows import classify_time_window
from src.analysis.aggregation import build_segment_pairs, aggregate_by_segment

TARGET_DATES = ["2026-10-22", "2026-10-23", "2026-10-25", "2026-10-26"]
dataset = build_prepared_dataset(RAW, STATIC, feed="A", target_dates=TARGET_DATES)
dataset = classify_time_window(dataset)

segments = build_segment_pairs(dataset)
print("Liczba odcinkow (wierszy) po oczyszczeniu:", len(segments))

aggregated = aggregate_by_segment(segments)
print("Liczba unikalnych kombinacji (odcinek x okno czasowe) po progu min. obserwacji:", len(aggregated))
print("\nNajwieksze mediany przyrostu opoznienia:")
print(aggregated.sort_values("median_increment", ascending=False).head(10).to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Liczba odcinkow (wierszy) po oczyszczeniu: 625422
Liczba unikalnych kombinacji (odcinek x okno czasowe) po progu min. obserwacji: 15546

Najwieksze mediany przyrostu opoznienia:
prev_stop_id next_stop_id day_type  time_of_day  median_increment  p85_increment  p95_increment  observation_count
        7650         7645  weekday morning_peak             206.5         251.50         282.50                  6
        8887        14017  weekend evening_peak             148.5         168.25         178.75                  6
        8887        14017  weekend     off_peak             139.0         232.50         270.90                 27
       17952        18176  weekday evening_peak             119.0         282.30         441.40                 15
        8886        13226  weekend morning_peak             109.5         147.25         149.75                  6
        8699        13194  weekday morning_peak             106.5         274.50         379.50                  6
       17914     

In [2]:
import pandas as pd

print(aggregated["observation_count"].describe())
print("\nRozklad liczby obserwacji (przedzialy):")
print(pd.cut(aggregated["observation_count"], bins=[0, 5, 10, 20, 50, 100, 1000]).value_counts().sort_index())

count    15546.000000
mean        39.584137
std         70.150746
min          5.000000
25%          9.000000
50%         19.000000
75%         41.000000
max       1804.000000
Name: observation_count, dtype: float64

Rozklad liczby obserwacji (przedzialy):
observation_count
(0, 5]          904
(5, 10]        3680
(10, 20]       3567
(20, 50]       4234
(50, 100]      1901
(100, 1000]    1250
Name: count, dtype: int64


In [3]:
solidne = aggregated[aggregated["observation_count"] >= 30]
print(f"Liczba kombinacji z n>=30: {len(solidne)}")
print("\nNajwieksze mediany przy solidnej probie:")
print(solidne.sort_values("median_increment", ascending=False).head(10).to_string(index=False))

Liczba kombinacji z n>=30: 5447

Najwieksze mediany przy solidnej probie:
prev_stop_id next_stop_id day_type time_of_day  median_increment  p85_increment  p95_increment  observation_count
       17914         9979  weekday    off_peak              42.0         117.60         160.20                 53
       17952        18176  weekday    off_peak              40.0         106.60         337.20                 49
       13945         8745  weekday    off_peak              38.0          55.15          64.80                 40
        8745        13943  weekday    off_peak              34.0          68.00          81.20                 37
       12318        12568  weekend    off_peak              32.5          51.60          69.95                 30
       13943        13941  weekday    off_peak              24.0          57.20          63.20                 37
       10086        10090  weekday    off_peak              18.0          39.00          47.00                 51
       10089  

In [4]:
aggregated_full = aggregate_by_segment(segments, min_observations=10)
aggregated_ranking = aggregate_by_segment(segments, min_observations=30)

print("Do pelnej mapy (próg 10):", len(aggregated_full), "kombinacji")
print("Do rankingu top N (próg 30):", len(aggregated_ranking), "kombinacji")

Do pelnej mapy (próg 10): 11408 kombinacji
Do rankingu top N (próg 30): 5447 kombinacji


## Wnioski

1. **Agregacja na odcinkach międzyprzystankowych działa poprawnie**: 625422 odcinki po
   oczyszczeniu (wykluczone `arrival_delay_extreme`, `uncertain_completion`, braki
   `stop_sequence`), zagregowane do 15546 unikalnych kombinacji (odcinek | dzień | pora)
   przy progu minimalnym 5.

2. **Odkryto efekt małej próby przy niskim progu**: Najwyższe mediany przy progu 5
   okazały się artefaktem. Przy n>=30 najwyższa mediana spada do 42s. Grupy z n<10 stanowią
   5.8% zbioru, mediana liczby obserwacji w całym zbiorze to 19.

3. **Przyjęto dwa progi do różnych celów:** 10 dla pełnej agregacji lub mapy (11408 kombinacji),
   30 dla rankingów "top N najgorszych odcinków" (5447 kombinacji). Najwyżej klasyfikowane odcinki przy tym progu
   są statystycznie wiarygodne, nie szumem.

4. **Przy progu 30 najwyżej klasyfikowane odcinki są zdominowane przez porę poza szczytem (`off_peak`) w dni
   robocze**: Wymaga dalszego zrozumienia w kolejnym etapie.

5. **Rozdzielczość przestrzenna ograniczona do par przystanków**: Przyrost opóźnienia jest
   przypisany do odcinka między dwoma kolejnymi, obserwowanymi przystankami, a nie do
   konkretnego punktu na trasie. Wykrycie miejsc niezależnych od rozstawu przystanków
   (np. konkretne skrzyżowanie w połowie długiego odcinka) wymaga ciągłego śledzenia pozycji
   (`vehicle_positions`, GPS) i geometrii trasy (`shapes.txt`). Odłożono to jako
   osobny, kolejny etap po zakończeniu preprocessingu.